# BANKING77 Intent Classification

Fine-grained intent detection on 13,083 customer service queries across 77 banking intents.

**Dataset:** [PolyAI BANKING77](https://github.com/PolyAI-LDN/task-specific-datasets)  
**Approach:** TF-IDF + Logistic Regression baseline → DistilBERT fine-tuning

## 1. Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

c:\Users\44753\OneDrive - University of St Andrews\Desktop\Freelance Portfolio\banking77_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## 2. Load Data

In [3]:
dataset = load_dataset('PolyAI/banking77')
print(dataset)

RuntimeError: Dataset scripts are no longer supported, but found banking77.py

In [ ]:
# Convert to DataFrames
label_names = dataset['train'].features['label'].names

train_df = dataset['train'].to_pandas()
test_df  = dataset['test'].to_pandas()

# Map integer labels to string names
train_df['intent'] = train_df['label'].map(lambda i: label_names[i])
test_df['intent']  = test_df['label'].map(lambda i: label_names[i])

print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows | Intents: {len(label_names)}')
train_df.head()

## 3. Exploratory Data Analysis

In [ ]:
# --- Class distribution ---
intent_counts = train_df['intent'].value_counts()

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(intent_counts)), intent_counts.values, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xticks([])
ax.set_xlabel('Intent (77 classes)', fontsize=12)
ax.set_ylabel('Training examples', fontsize=12)
ax.set_title('Class Distribution — BANKING77 Training Set', fontsize=14)
ax.axhline(intent_counts.mean(), color='coral', linestyle='--', label=f'Mean = {intent_counts.mean():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Examples per intent  min={intent_counts.min()}  max={intent_counts.max()}  mean={intent_counts.mean():.1f}')

In [ ]:
# --- Query length distribution ---
train_df['n_words'] = train_df['text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(train_df['n_words'], bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of words', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Query Length Distribution', fontsize=14)
plt.tight_layout()
plt.show()

print(train_df['n_words'].describe().round(1))

In [ ]:
# --- Sample queries per intent ---
sample_intents = ['lost_or_stolen_card', 'exchange_rate', 'top_up_failed', 'verify_my_identity']
for intent in sample_intents:
    samples = train_df.loc[train_df['intent'] == intent, 'text'].sample(2, random_state=SEED).tolist()
    print(f'\n[{intent}]')
    for s in samples:
        print(f'  • {s}')

## 4. Baseline: TF-IDF + Logistic Regression

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, sublinear_tf=True)
X_train = tfidf.fit_transform(train_df['text'])
X_test  = tfidf.transform(test_df['text'])

lr = LogisticRegression(max_iter=1000, C=5, random_state=SEED)
lr.fit(X_train, train_df['label'])

y_pred_baseline = lr.predict(X_test)
acc_baseline = accuracy_score(test_df['label'], y_pred_baseline)
f1_baseline  = f1_score(test_df['label'], y_pred_baseline, average='macro')

print(f'Baseline  Accuracy: {acc_baseline:.4f}  Macro-F1: {f1_baseline:.4f}')

## 5. DistilBERT Fine-Tuning

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

hf_train = dataset['train'].map(tokenize, batched=True)
hf_test  = dataset['test'].map(tokenize, batched=True)

hf_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
hf_test.set_format('torch',  columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
model.to(device);

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

training_args = TrainingArguments(
    output_dir='./distilbert-banking77',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=50,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_test,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Evaluation

In [ ]:
results = trainer.evaluate()
print(f"DistilBERT  Accuracy: {results['eval_accuracy']:.4f}  Macro-F1: {results['eval_macro_f1']:.4f}")
print(f"Baseline    Accuracy: {acc_baseline:.4f}  Macro-F1: {f1_baseline:.4f}")

In [ ]:
# Full classification report
preds_output = trainer.predict(hf_test)
y_pred_bert  = np.argmax(preds_output.predictions, axis=-1)
y_true       = test_df['label'].values

print(classification_report(y_true, y_pred_bert, target_names=label_names))

In [ ]:
# Confusion matrix — top 20 most confused intent pairs
cm = confusion_matrix(y_true, y_pred_bert)

# Zero out diagonal (correct predictions), find the worst pairs
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)

top_n = 20
flat_idx = np.argsort(cm_off.ravel())[::-1][:top_n]
rows, cols = np.unravel_index(flat_idx, cm_off.shape)

confused_labels = sorted(set(rows.tolist() + cols.tolist()))
sub_cm = cm[np.ix_(confused_labels, confused_labels)]
sub_names = [label_names[i] for i in confused_labels]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    sub_cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=sub_names, yticklabels=sub_names, ax=ax,
    linewidths=0.5,
)
ax.set_title('Confusion Matrix — Most Confused Intent Pairs', fontsize=14)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 7. Inference Demo

In [ ]:
def predict_intent(query: str) -> str:
    """Return the predicted banking intent for a customer query."""
    inputs = tokenizer(query, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id = logits.argmax(-1).item()
    confidence = torch.softmax(logits, dim=-1)[0, pred_id].item()
    return label_names[pred_id], confidence


example_queries = [
    "I lost my card, what should I do?",
    "Why hasn't my top-up appeared yet?",
    "What's the exchange rate for USD to EUR?",
    "My card payment was charged twice",
    "How do I change my PIN?",
    "I want to cancel a transfer I just made",
    "Can I use my card in Japan?",
    "How long does it take to get a physical card?",
]

print(f'{'Query':<55} {'Predicted Intent':<45} Confidence')
print('-' * 110)
for q in example_queries:
    intent, conf = predict_intent(q)
    print(f'{q:<55} {intent:<45} {conf:.2%}')